# Qriton HLM — Quickstart

Program neural networks by shaping energy landscapes.  
No training. No fine-tuning. Direct surgery on the weight matrix.

```bash
pip install qriton-hlm[jupyter]
```

In [ ]:
import torch
from qriton_hlm import BasinSurgeon, compute_energy, find_basins

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'PyTorch {torch.__version__}, device={DEVICE}')

## 1. Create a synthetic Hopfield network
No checkpoint needed — raw W matrix to understand fundamentals.

In [ ]:
d_model = 128
torch.manual_seed(42)
W = torch.randn(d_model, d_model, device=DEVICE) * 0.01
W = (W + W.T) / 2  # symmetric

surgeon = BasinSurgeon.from_W(W, device=DEVICE)
print(f'W: {W.shape}, spectral={torch.linalg.norm(W, ord=2).item():.4f}')

## 2. Survey — discover existing basins

In [ ]:
survey = surgeon.survey(layer=0)
print(f'{survey["num_basins"]} basins found ({survey["elapsed"]:.2f}s)\n')
for i in range(min(10, survey['num_basins'])):
    print(f'  B{i:2d}  E={survey["energies"][i]:+.4f}  pop={survey["populations"][i]}')

## 3. Inject — program a new attractor

In [ ]:
result = surgeon.inject(layer=0, seed=42, strength=0.1)
print(f'Before: existed={result["existed_before"]} cos={result["cos_before"]:.4f}')
print(f'After:  exists={result["exists_after"]}  cos={result["cos_after"]:.4f}')
if result['exists_after'] and not result['existed_before']:
    print('\n>> New basin programmed!')

## 4. Verify — confirm stability

In [ ]:
v = surgeon.verify(layer=0, seed=42)
print(f'Basin: {v["is_basin"]}  cos={v["cos"]:.6f}  iters={v["iters"]}  E={v["energy"]:.4f}')

## 5. Measure the damage

In [ ]:
diff = surgeon.diff(layer=0)
if diff:
    print(f'Relative change: {diff["relative_pct"]:.2f}%')
    print(f'Spectral: {diff["spectral_orig"]:.4f} -> {diff["spectral_curr"]:.4f}')

## 6. Remove — destroy the attractor

In [ ]:
r = surgeon.remove(layer=0, seed=42, strength=0.2)
print(f'Still exists: {r["exists_after"]} cos={r["cos_after"]:.4f}')

## 7. Restore

In [ ]:
surgeon.restore(layer=0)
print('Restored to original W.')

---
**Next:** `02_concept_surgery.ipynb` — capture concepts from text and inject them